# 01 — Data preparation and splitting

Clean the data, create the binary label, create the stratified full split, and create the fixed 200k/20k subset. Existing split files are treated as immutable and are checked rather than replaced.

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)
RAW = ROOT / "data/raw/train.csv"
CLEANED = ROOT / "data/processed/cleaned.csv"
FULL = ROOT / "data/splits/full"
SUBSET = ROOT / "data/splits/200k"
SEED = 42
EXPECTED_ROWS = {
    "full_train": 1_424_657,
    "full_validation": 178_082,
    "full_test": 178_083,
    "subset_train": 200_000,
    "subset_validation": 20_000,
}
assert RAW.exists()

def check_split(path, expected_rows):
    frame = pd.read_csv(path)
    assert len(frame) == expected_rows
    assert frame["id"].is_unique
    assert frame["target"].ge(0.5).astype("int8").equals(frame["label"].astype("int8"))
    return frame

## Clean and label

In [4]:
data = pd.read_csv(RAW, usecols=["id", "target", "comment_text"]).dropna(
    subset=["id", "target", "comment_text"]
)
data["comment_text"] = data["comment_text"].astype(str).str.strip()
data = data[data["comment_text"].ne("")].drop_duplicates("comment_text").copy()
data["label"] = data["target"].ge(0.5).astype("int8")
assert data["id"].is_unique
CLEANED.parent.mkdir(parents=True, exist_ok=True)
if not CLEANED.exists():
    data.to_csv(CLEANED, index=False)
else:
    saved = pd.read_csv(CLEANED)
    assert len(saved) == len(data)
    assert saved["id"].is_unique
    assert saved["target"].ge(0.5).astype("int8").equals(saved["label"].astype("int8"))
print(f"Cleaned rows: {len(data):,}")

Cleaned rows: 1,780,822


## Stratified full split

In [6]:
data = pd.read_csv(CLEANED)
FULL.mkdir(parents=True, exist_ok=True)
full_paths = {name: FULL / f"{name}.csv" for name in ["train", "validation", "test"]}
if not all(path.exists() for path in full_paths.values()):
    assert not any(path.exists() for path in full_paths.values())
    train, held_out = train_test_split(
        data, test_size=0.20, random_state=SEED, stratify=data["label"]
    )
    validation, test = train_test_split(
        held_out, test_size=0.50, random_state=SEED, stratify=held_out["label"]
    )
    for name, frame in {"train": train, "validation": validation, "test": test}.items():
        frame[["id", "comment_text", "target", "label"]].sort_values("id").to_csv(
            full_paths[name], index=False
        )
full = {
    "train": check_split(full_paths["train"], EXPECTED_ROWS["full_train"]),
    "validation": check_split(full_paths["validation"], EXPECTED_ROWS["full_validation"]),
    "test": check_split(full_paths["test"], EXPECTED_ROWS["full_test"]),
}
assert not (set(full["train"]["id"]) & set(full["validation"]["id"]))
assert not (set(full["train"]["id"]) & set(full["test"]["id"]))
assert not (set(full["validation"]["id"]) & set(full["test"]["id"]))
print({name: len(frame) for name, frame in full.items()})

{'train': 1424657, 'validation': 178082, 'test': 178083}


## Fixed 200k/20k subset and checks

In [8]:
SUBSET.mkdir(parents=True, exist_ok=True)
subset_paths = {name: SUBSET / f"{name}.csv" for name in ["train", "validation"]}
if not all(path.exists() for path in subset_paths.values()):
    assert not any(path.exists() for path in subset_paths.values())
    for name, rows in [("train", 200000), ("validation", 20000)]:
        selected = train_test_split(
            full[name], train_size=rows, random_state=SEED, stratify=full[name]["label"]
        )[0]
        selected.sort_values("id").to_csv(subset_paths[name], index=False)
subset = {
    "train": check_split(subset_paths["train"], EXPECTED_ROWS["subset_train"]),
    "validation": check_split(subset_paths["validation"], EXPECTED_ROWS["subset_validation"]),
}
assert set(subset["train"]["id"]).issubset(set(full["train"]["id"]))
assert set(subset["validation"]["id"]).issubset(set(full["validation"]["id"]))
print("PASS | Full split and fixed subset checks completed.")

PASS | Full split and fixed subset checks completed.
